In [0]:
###TRANSFORMASI DATA DARI BRONZE KE SILVER

from pyspark.sql import functions as F

catalog = "databrics_ecommerce"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver")

def save(df, name):
    df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.silver.{name}")
    print(name, df.count())

# 1. orders: ubah kolom tanggal jadi timestamp
orders = (spark.table(f"{catalog}.bronze.orders")
    .dropDuplicates(["order_id"])
    .withColumns({c: F.to_timestamp(c) for c in [
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date",
        "order_estimated_delivery_date"]})
    .filter(F.col("order_id").isNotNull()))
save(orders, "orders")

# 2. order_items: ubah harga jadi angka
items = (spark.table(f"{catalog}.bronze.order_items")
    .dropDuplicates(["order_id", "order_item_id"])
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("freight_value", F.col("freight_value").cast("double"))
    .filter(F.col("price") >= 0))
save(items, "order_items")

# 3. customers
customers = (spark.table(f"{catalog}.bronze.customers")
    .dropDuplicates(["customer_id"])
    .withColumn("customer_city", F.initcap("customer_city")))
save(customers, "customers")

# 4. products: perbaiki typo kolom bawaan dataset
products = (spark.table(f"{catalog}.bronze.products")
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
    .dropDuplicates(["product_id"]))
save(products, "products")###CEK DENGAN ASSERT APAKAH DATANYA SUDAH SESUAI DAN TIDAK ADA ERROR

assert spark.table(f"{catalog}.silver.orders").filter("order_id IS NULL").count() == 0
assert spark.table(f"{catalog}.silver.order_items").filter("price < 0").count() == 0

In [0]:
###CEK DENGAN ASSERT APAKAH DATANYA SUDAH SESUAI DAN TIDAK ADA ERROR

assert spark.table(f"{catalog}.silver.orders").filter("order_id IS NULL").count() == 0
assert spark.table(f"{catalog}.silver.order_items").filter("price < 0").count() == 0

In [0]:
###CEK SEMUA KOLOM APAKAH ADA DATA NULL, DATA STRING NORMAL KALO NILAINYA NULL

from pyspark.sql import functions as F

catalog = "databrics_ecommerce"
tables = [t.name for t in spark.catalog.listTables(f"{catalog}.silver")]

rows = []
for t in tables:
    df = spark.table(f"{catalog}.silver.{t}")
    total = df.count()
    nulls = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
    ]).first().asDict()
    for col, n in nulls.items():
        if n > 0:
            rows.append((t, col, n, total, round(n / total * 100, 2)))

report = spark.createDataFrame(rows, ["tabel", "kolom", "jumlah_null", "total_baris", "persen_null"])
display(report.orderBy("tabel", F.desc("persen_null")))

In [0]:
### CEK KOLOM YANG PENTING APAKAH ADA NULL

must_not_null = {"orders": ["order_id", "customer_id"],
                 "order_items": ["order_id", "product_id"]}

for t, cols in must_not_null.items():
    for c in cols:
        n = spark.table(f"{catalog}.silver.{t}").filter(F.col(c).isNull()).count()
        assert n == 0, f"{t}.{c} punya {n} null"

In [0]:
### MEMASUKKAN DAN MEMBERSIHKAN TABLE SELLER DAN TRANSLATION KE DALAM SILVER 

from pyspark.sql import *

for t in ["sellers", "product_category_name_translation"]:
    (spark.table(f"databrics_ecommerce.bronze.{t}")
     .dropDuplicates()
     .write.format("delta").mode("overwrite")
     .saveAsTable(f"databrics_ecommerce.silver.{t}"))